# Taller Congreso CECFA
Pía Amigo, 04 de agosto de 2026



## Redshift fotométrico usando regresión lineal y una red neuronal totalmente conectada

Este notebook está diseñado para guiarte paso a paso en el desarrollo del mini-proyecto, siguiendo cada celda. En algunos casos, **se destacan en negrita tareas específicas** que deberás completar. Puedes revisar las soluciones provistas para comparar tus resultados.

---

### ¿Qué es el redshift?

Hoy sabemos que las estrellas y galaxias se encuentran a distintas distancias, a veces a miles de millones de años luz. Medir estas distancias es crucial para construir un mapa tridimensional del Universo y para convertir el brillo aparente en luminosidad intrínseca, revelando la verdadera energía emitida por las fuentes astrofísicas. Además, como la luz viaja a velocidad finita, la distancia corresponde a un tiempo hacia el pasado, lo que nos permite estudiar la historia del Universo.

Más aún, Edwin Hubble descubrió en 1929 que el Universo se está *expandiendo*, y que mientras más lejos está una galaxia, más rápido se aleja de nosotros. Esto produce un efecto Doppler en la luz que recibimos de objetos lejanos. En astrofísica, este corrimiento Doppler se conoce como **redshift** \(z\), y puede expresarse en función de la longitud de onda observada y emitida como:

$1 + z = \frac{\lambda_{\mathrm{obs}}}{\lambda_{\mathrm{em}}}$

La mejor manera de determinar redshifts es calcular el desplazamiento de líneas conocidas en un *espectro*, que corresponde a la emisión de luz en función de la longitud de onda. Sin embargo, los espectros pueden ser de baja calidad si los objetos están demasiado lejos, y obtener datos espectroscópicos de alta calidad es costoso. La fotometría, en cambio, es mucho más barata y fácil de obtener. En este caso, disponemos de la emisión promedio (o brillo) en rangos de longitudes de onda (conocidos como bandas o filtros).

<img src="https://skyserver.sdss.org/dr14/en/get/SpecById.ashx?id=1833056915844786176" alt="espectro" width="500">

Por lo tanto, podemos aprovechar datos fotométricos para muestras mucho más grandes de galaxias en el Universo, y los próximos surveys están pensados para observar una fracción significativa (>10%) de todas las galaxias del Universo. Ser capaces de derivar redshifts confiables para estas galaxias es fundamental para la astronomía.


### El dataset

El conjunto de datos utilizado en este notebook proviene de este artículo de [Zhou et al. (2019)](https://academic.oup.com/mnras/article/488/4/4565/5538813). Corresponde a una compilación de varios surveys, como DEEP2, DEEP3 y 3D-HST. Los datos reproducen la cobertura en longitud de onda y la profundidad del futuro Observatorio Vera Rubin, que se espera provea fotometría en seis bandas, desde el ultravioleta cercano hasta el infrarrojo cercano (u, g, r, i, z e y), para aproximadamente 20 mil millones de galaxias, abarcando una fracción considerable del volumen del Universo.

Los resultados obtenidos en el artículo se muestran en la siguiente figura:

<img src="https://oup.silverchair-cdn.com/oup/backfile/Content_public/Journal/mnras/488/4/10.1093_mnras_stz1866/1/stz1866fig5.jpeg?Expires=1788874138&Signature=IuMCFAGV-3U3VVZINVIOV2iBOKui9QQmBV4V1xXf3diaEucIqyNJtddqnTwmvxzoz80xnZqablo1B6YMiPe79oVpyUugha9Z~CcEA7C-6UlFP-TEGup7iU7UpIg9Nfaa7vrl9oydFncpNlAtg4HW76Sz-mvYX7o5VYdgFKUXz4H2BD1RZn5jxe5atodK66jyDieIhM2z5UTglDMY-HMUZwwuESnZjIbPXwiYoPMz~ulYyjGKgne1ppznvvxUONWLwinMEXDSEzKsY1L3rGe4wl0lE3es1VrmxmQVNAuLBYMaWS3TtMdviYvK5PvxJdRwpqpaA495i8lPk8TA7NqAtw__&Key-Pair-Id=APKAIE5G5CRDK6RD3PGA" alt="photo_z_paper" width="500">

donde $\sigma_{\mathrm{NMAD}}$ es la desviación absoluta mediana normalizada de los residuos y $\eta$ es la fracción de *outliers*, definida como los objetos que cumplen

$\frac{|(z_{\mathrm{spec}} - z)|}{(1 + z_{\mathrm{spec}})} > 0.15$

En el artículo, los valores reportados son $\sigma_{\mathrm{NMAD}} = 0.0174$ y $\eta = 4.54\%$ (fracción de *outliers*).



## Features y target

En machine learning trabajamos con dos tipos de variables:

- **Features**:  
  Son las cantidades que usamos como entrada del modelo.  
  En este caso, corresponden a las magnitudes fotométricas en distintas bandas, es decir, la información observable de cada galaxia.

- **Target**:  
  Es la cantidad que queremos predecir.  
  En este notebook, el target es el redshift espectroscópico `zhelio`, una magnitud física que no conocemos para todas las galaxias.

Cada fila del conjunto de datos representa un objeto astronómico:

$(\text{features}) \;\longrightarrow\; (\text{target})$

El objetivo del modelo es aprender una función que aproxime esta relación:

$f(\text{features}) \;\approx\; \text{target}$

Es decir, aprender a predecir el redshift a partir de la información fotométrica.


Ahora comenzaremos a trabajar con el dataset. Debemos cargar los archivos `sel_features.csv` y `sel_target.csv`, que contiene el set de datos descrito anteriormente


In [ ]:
#importamos las librerías necesarias
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.model_selection import train_test_split
from sklearn.model_selection import KFold
from sklearn.utils import shuffle
from sklearn.preprocessing import StandardScaler

In [ ]:
import matplotlib
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', 100)
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_colwidth', 150)

font = {'size'   : 10}
matplotlib.rc('font', **font)
matplotlib.rc('xtick', labelsize=8)
matplotlib.rc('ytick', labelsize=8)
matplotlib.rcParams['figure.dpi'] = 300

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
#!pip install astropy


Leeremos los datos de caracteristicas (las magnitudes) y targets (el redshift espectroscópico o real)

In [ ]:
X = pd.read_csv('/content/drive/MyDrive/taller_cecfa_2026/sel_features.csv', sep = '\t')
y = pd.read_csv('/content/drive/MyDrive/taller_cecfa_2026/sel_target.csv')

**¿Cuántos objetos tiene el dataset?**

In [ ]:
X

In [ ]:
y

Puede explorar los datos para responder algunas de estas preguntas
- **¿Qué rangos tienen las distintas features?**
- **¿Cómo se distribuye una magnitud (por ejemplo `i_apercor`)?**
- **¿Existe alguna relación visible entre un color (por ejemplo `g-r`) y el redshift?**
- **¿Cómo se distribuyen los objetos en distintos rangos de $z$?**

#### **Explorar los datos en un proyecto de ML es muy importante, para poder interpretar los resultados después del modelo**

## Modelo base: Regresión lineal

Comenzaremos con el modelo más simple posible: una **regresión lineal**.  
En este caso, para poder hacer una predicción, asumimos que el redshift puede aproximarse como una combinación lineal de las magnitudes fotométricas:

$z \approx a_0 + a_1 x_1 + a_2 x_2 + \dots + a_n x_n$

Este modelo es deliberadamente simple. Su objetivo no es ser el mejor, sino servir como **línea base** (*baseline*):

- Nos permite introducir la idea de *entrenar un modelo* a partir de datos.  
- Ilustra claramente que *machine learning* consiste en ajustar parámetros para minimizar un error.  
- Nos da un punto de comparación para evaluar luego modelos más flexibles, como una red neuronal.

Antes de entrenar cualquier modelo, debemos dividir nuestros datos en un conjunto de entrenamiento y uno de prueba.


## Separación en entrenamiento y prueba

En machine learning no entrenamos un modelo usando todos los datos disponibles.  
En su lugar, dividimos el conjunto de datos en dos partes:

- **Training set (conjunto de entrenamiento):**  
  Se utiliza para ajustar los parámetros del modelo. Es aquí donde el algoritmo “aprende”.

- **Test set (conjunto de prueba):**  
  Se mantiene separado y **no** se usa durante el entrenamiento. Sirve para evaluar qué tan bien el modelo generaliza a datos que nunca ha visto.

Esta separación es fundamental: si evaluáramos el modelo con los mismos datos que usamos para entrenarlo, podríamos obtener un resultado artificialmente bueno, sin saber si el modelo realmente funciona en casos nuevos.

En este notebook usaremos una división típica de 70% para entrenamiento y 30% para prueba.


In [ ]:
from sklearn.model_selection import train_test_split

# Separar en entrenamiento y prueba (70% / 30%)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

print("Tamaño del set de entrenamiento:", X_train.shape)
print("Tamaño del set de prueba:", X_test.shape)


**Prueba cambiar la fracción del set de entrenamiento. Crees que esto afectaría a tu modelo?**

Ahora implementamos la regresión lineal, importando la librería de `sklearn`

Scikit-learn es la librería de Machine Learning en Python.

[`sklearn`](https://scikit-learn.org/stable/index.html) provee implementaciones eficientes y bien documentadas de:

- Modelos de regresión y clasificación  
- Algoritmos de clustering  
- Métodos de reducción de dimensionalidad  
- Herramientas para preprocesamiento, validación y evaluación de modelos  

Su diseño es coherente y simple: casi todos los modelos siguen la misma interfaz:

1. Crear el modelo  
2. Entrenarlo con `.fit(X_train, y_train)`  
3. Predecir con `.predict(X_test)`  

Esto permite cambiar de algoritmo con muy pocas líneas de código, manteniendo el mismo flujo de trabajo.

- Guía introductoria: https://scikit-learn.org/stable/tutorial/basic/tutorial.html
- Galería de ejemplos: https://scikit-learn.org/stable/auto_examples/index.html



In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

In [ ]:
# entrenamos el  modelo lineal
lin = LinearRegression()
lin.fit(X_train, y_train) # usamos los datos de entrenamiento

# ajustamos el modelo a las características del set de prueba y generamos la predicción de redshift
y_pred_lin = lin.predict(X_test)

# métrica para evaluar el desempeño del modelo
rmse_lin = np.sqrt(mean_squared_error(y_test, y_pred_lin))
print(f"RMSE (Regresión lineal): {rmse_lin:.4f}")

## Métrica de evaluación: RMSE

Para evaluar el desempeño del modelo usaremos el **RMSE** (Root Mean Squared Error), definido como:

$\mathrm{RMSE} = \sqrt{\frac{1}{N} \sum_{i=1}^{N} (y_{\text{true},i} - y_{\text{pred},i})^2}$

Esta métrica mide el error promedio entre el valor real y la predicción, en las mismas unidades que el redshift.  
Un RMSE menor indica mejores predicciones.


Una forma intuitiva de evaluar el modelo es comparar visualmente sus predicciones con los valores reales, es decir, graficar $y_{\text{true}}$ frente a $y_{\text{pred}}$.  
En el caso ideal, todos los puntos deberían alinearse sobre la recta \(y = x\): eso indicaría que el modelo predice exactamente los valores verdaderos. Mientras más cerca estén los puntos de esa diagonal, mejor es el desempeño del modelo


In [ ]:
plt.figure(figsize=(3,3))
plt.scatter(y_test, y_pred_lin, s=5, alpha=0.5)
plt.plot([y_test.min(), y_test.max()],
         [y_test.min(), y_test.max()],
         'r--', lw=2)
plt.xlabel(r"$z_{\mathrm{true}}$")
plt.ylabel(r"$z_{\mathrm{pred}}$")
plt.title("Regresión lineal")
plt.tight_layout()
plt.show()

y podemos calcular las métricas que usan en el paper

In [ ]:
len(np.where(np.abs(y_test-y_pred_lin)>0.15*(1+y_test))[0])/len(y_test)


In [ ]:
1.48*np.median(np.abs(y_test-y_pred_lin)/(1 + y_test))

- **Comenta este resultado. Te parece que el modelo de regresión lineal es apropiado para este problema?**
- **¿Qué ocurre si eliminas una de las *features* (por ejemplo `u_apercor`)?**

## Redes neuronales

## Red neuronal: un modelo más flexible

La regresión lineal asume que el redshift puede escribirse como una combinación lineal de las features.  
Sin embargo, la relación entre colores fotométricos y redshift es inherentemente **no lineal** y más compleja.

Para capturar este tipo de relaciones utilizaremos una **red neuronal totalmente conectada** (Multi-Layer Perceptron, MLP).  
Una red neuronal no es más que una función más flexible, construida como una composición de funciones simples:

$\text{features} \;\longrightarrow\; \text{capas} \;\longrightarrow\; z_{\text{pred}}$

El entrenamiento sigue exactamente la misma idea que antes:

- Definimos una función de pérdida (error).  
- Ajustamos los parámetros del modelo para minimizarla.  
- Repetimos este proceso hasta que el error disminuya.

Lo que cambia respecto a la regresión lineal no es el principio, sino la **forma de la función** que estamos ajustando.  
Veamos ahora cómo implementar una red neuronal sencilla para este mismo problema.


Para implementar una red neuronal, usaremos [TensorFlow](https://www.tensorflow.org/?hl=es-419). Esta librería es una biblioteca muy utilizada para el desarrollo de modelos de Deep Learning. Es una plataforma de código abierto desarrollada por Google. Permite programar en varios lenguajes, como C++, Java, Python, entre otros.

Keras es una API de alto nivel (*Application Programming Interface*) construida sobre TensorFlow (o Theano, otra biblioteca de Deep Learning). Es específica de Python, y podemos pensarla como el equivalente de la librería `sklearn` pero para redes neuronales. Es menos general y menos personalizable, pero muy amigable para el usuario y comparativamente más simple que usar TensorFlow directamente. En este notebook utilizaremos Keras con TensorFlow como back-end.


In [ ]:
import tensorflow as tf

In [ ]:
import keras

from keras.models import Sequential #El modelo se construye agregando capas una tras otra.

from keras.layers import Dense, Input #capas totalmente conectadas: cada salida está conectada con cada entrada

from keras.layers import Dropout #para regularización

En el caso de las redes neuronales, es útil introducir un **conjunto de validación** adicional. A diferencia de modelos simples, las redes tienen muchos parámetros y pueden adaptarse demasiado bien a los datos de entrenamiento, aprendiendo incluso el ruido. El conjunto de validación permite monitorear el desempeño del modelo durante el entrenamiento en datos que no está usando para ajustar sus pesos. De esta forma, podemos detectar si el modelo está empezando a **sobreajustar** (cuando la pérdida en entrenamiento sigue bajando, pero la de validación deja de mejorar) y tener una idea más realista de su capacidad de generalización. El conjunto de prueba, en cambio, se reserva exclusivamente para la evaluación final del modelo.


In [ ]:
# Primero separamos test (20%), estratificado por clase
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

# Del resto, separamos train/val (80/20 de lo que queda)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.2,  random_state=10
)

print(X_train.shape, X_val.shape, X_test.shape)


Antes de entrenar nuestros modelos, es importante preguntarnos algo simple:

> ¿Tiene sentido que una variable con valores del orden de $(10^{-2})$ y otra del orden de $(10^{2})$ entren a un modelo “tal como están”?

En nuestro caso, las features (magnitudes) pueden tener rangos muy distintos.  
Muchos algoritmos de machine learning, y en particular las redes neuronales, se basan en optimización mediante **gradiente descendiente**, y son sensibles a estas diferencias de escala.

Si una variable domina numéricamente sobre las otras, el modelo puede:

- Aprender más lento,  
- Tener dificultades para converger,  
- Dar demasiado peso a ciertas features solo por su escala.

Escalar los datos significa transformar cada feature para que tenga una escala comparable (por ejemplo, media 0 y desviación estándar 1).  
Esto no cambia la información física, pero sí facilita que el modelo aprenda de manera más estable y eficiente.

Escalaremos nuestros datos antes de implementar la red. Usaremos [`StandardScaler`](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html) de `sklearn`


In [ ]:
scaler = StandardScaler()

scaler.fit(X_train)

In [ ]:
Xst_train = scaler.transform(X_train)
Xst_val = scaler.transform(X_val)
Xst_test = scaler.transform(X_test)

En un problema de regresión, la red neuronal debe entregar un valor continuo como salida.  
Por esta razón, en la **capa de salida** utilizamos una activación lineal, y escogemos una función de pérdida apropiada para variables continuas, en este caso MSE

En nuestro caso, cada galaxia está descrita por **seis *features*** fotométricas, por lo que la **capa de entrada** tendrá seis neuronas.

Como punto de partida, construiremos una red neuronal simple pero suficientemente flexible:

- Dos capas ocultas con 100 neuronas cada una.  
- Función de activación `relu` en las capas ocultas, que introduce no linealidad.  
- Una única neurona de salida con activación `linear`, que devuelve el redshift predicho.  

El modelo se define de la siguiente manera:


In [ ]:
model = Sequential()

optimizer = tf.keras.optimizers.Adam(learning_rate=0.001) #similar al gradiente descendiente

# Agrega una capa de entrada y especifica el tamaño (generalmente es el tamaño del set de features)

model.add(Input(shape=(6,)))


# Capas ocultas y se especifica el tamaño (número de neuronas)

model.add(Dense(100, activation='relu'))
#model.add(Dropout(0.2)) #regularizacion
model.add(Dense(100, activation='relu'))
#model.add(Dropout(0.2))#regularizacion

# Agrega capa de salida

model.add(Dense(1, activation='linear'))

model.compile(loss='mse', optimizer=optimizer, metrics=['mse'])


**Puedes cambiar los parámetros de esta red y ver cómo cambia el desempeño del modelo y cómo afecta cada uno de estos parámetros**

Comenzamos con 100 epochs y un tamaño de batch de 300.


In [ ]:
mynet = model.fit(Xst_train, y_train, validation_data= (Xst_val, y_val), epochs=100, batch_size=300)

In [ ]:
results = model.evaluate(Xst_test, y_test)
print('MSE:', results)

Podemos visualizar cómo evoluciona la función de pérdida a lo largo del entrenamiento.  
Este gráfico nos muestra cómo la red va reduciendo el error en cada epoch, y nos permite verificar si el modelo está aprendiendo de manera estable o si aparecen problemas como una convergencia lenta o sobreajuste.


In [ ]:
plt.figure(figsize=(3,3))

plt.plot(mynet.history['loss'], label = 'train')
plt.plot(mynet.history['val_loss'],'-.m', label = 'validation')
plt.ylabel('Loss', fontsize = 8)
plt.xlabel('Epoch', fontsize = 8)
plt.legend(loc='upper right', fontsize = 8)
plt.legend(fontsize = 12);

**Qué observa de este gráfico?** **Podemos decir si el modelo está sobreajustando o subajustando?**

Comparamos las predicciones con los valores reales

In [ ]:
plt.figure(figsize=(3,3))

plt.xlabel('True redshift', fontsize = 14)
plt.ylabel('Estimated redshift', fontsize = 14)

plt.scatter(y_test, model.predict(Xst_test), s =10, c = 'teal');

plt.xlim(0,2)
plt.ylim(0,2)
plt.tight_layout()
#plt.savefig('Photoz_NN_scatter.png')

Generamos las predicciones para calcular la métrica que usa el paper y comparar

In [ ]:
ypred_NN = model.predict(Xst_test)

Calculamos fracción de outliers

In [ ]:
len(np.where(np.abs(y_test-ypred_NN)>0.15*(1+y_test))[0])/len(y_test)

y Normalized Median Absolute Deviation (NMAD)

In [ ]:
1.48*np.median(np.abs(y_test-ypred_NN)/(1 + y_test))

**Preguntas**:

- **¿Cómo se compara esto con el resultado del paper?**
- **¿Cómo cambia el resultado si usas una red más pequeña, por ejemplo `(32, 16)`?**
- **¿Y si usas una red más grande, como `(64, 64, 32)`?**
- **¿Qué ocurre al cambiar la función de activación de `relu` a `tanh`?**
- **¿Qué efecto tiene aumentar o disminuir el número de *epochs*?**
- **¿Cómo cambia el comportamiento al modificar el `batch_size`?**

Hiperparámetros y diseño del modelo

Además de los parámetros que la red aprende automáticamente (los pesos), una red neuronal tiene una serie de **hiperparámetros** que nosotros debemos elegir:

- Número de capas ocultas  
- Número de neuronas por capa  
- Función de activación  
- *Learning rate* del optimizador  
- Tamaño del *batch*  
- Número de *epochs*  

Estas elecciones determinan la forma y la capacidad del modelo. No existe una configuración universalmente óptima, el mejor conjunto de hiperparámetros depende del problema, de los datos y del objetivo científico.

En proyectos reales, estos hiperparámetros se **optimizan** de manera sistemática (por ejemplo, mediante búsqueda en grilla, búsqueda aleatoria o métodos más avanzados).  
En este taller, en cambio, los exploramos manualmente para entender cómo afectan el comportamiento y el desempeño del modelo.


#### **Reflexión final**
- **¿Qué modelo parece generalizar mejor?**
- **¿Qué tan sensible es el resultado a las decisiones de diseño del modelo?**
- **¿Qué features parecen más relevantes para predecir el redshift?**